<!--nav--> [🗺 Learning path](README.md) · **13/33** · ◀ [Simple MultiGPU Alignment Showdown](./Simple_MultiGPU_Alignment_Showdown.ipynb) · [LLM as Judge Evaluation](./LLM_as_Judge_Evaluation.ipynb) ▶

# GRPO: Teaching a Model to Reason with Pure RL

## The Breakthrough Behind DeepSeek-R1

In January 2025, DeepSeek released R1 — a model that learned to **think step-by-step** using nothing but
reinforcement learning. No chain-of-thought training data. No reward model. Just a simple signal:
**"is the answer correct?"**

The algorithm that made this possible: **GRPO (Group Relative Policy Optimization)**.

### What makes GRPO special

```
PPO (old way):                          GRPO (new way):
┌──────────┐  ┌──────────┐             ┌──────────┐
│  Policy  │  │  Critic  │             │  Policy  │     No critic needed!
│  Model   │  │  Model   │             │  Model   │     Save ~50% memory
│  (7B)    │  │  (7B)    │             │  (7B)    │
└────┬─────┘  └────┬─────┘             └────┬─────┘
     │              │                        │
     │   Advantage = R - V(s)                │   Generate G completions per prompt
     │   (needs critic)                      │   Advantage = (r_i - mean) / std
     │              │                        │   (group statistics replace critic)
     v              v                        v
 Total: ~14B params to train            Total: ~7B params to train
```

### This Notebook

```
Step 1: Understand GRPO — math, intuition, diagrams
Step 2: Setup — model, dataset, reward functions
Step 3: Train — watch the model learn to reason
Step 4: Evaluate — before/after on math problems
Step 5: Visualize — training curves, reasoning emergence
```

| Component | Details |
|-----------|--------|
| **Model** | Qwen2.5-1.5B (sweet spot for T4 GPUs) |
| **Dataset** | GSM8K — 7,473 grade-school math problems |
| **Reward** | Binary accuracy (correct=1, wrong=0) + format bonus |
| **Algorithm** | GRPO with 4 completions per prompt |
| **Platform** | Kaggle T4 x2 (free) or Colab T4 |

---
**Runtime:** T4 GPU (Kaggle or Colab)

---
# Part 1: Understanding GRPO

## How GRPO Works — Step by Step

```
For each training step:

  1. SAMPLE a batch of prompts
     ┌─────────────────────────────────────┐
     │  "What is 24 × 15?"                 │
     └─────────────────────────────────────┘

  2. GENERATE G completions per prompt (G=4 in our case)
     ┌────────────────┐  ┌────────────────┐
     │ Completion 1:  │  │ Completion 2:  │
     │ 24×15 = 24×10  │  │ 24×15 = 360    │
     │ + 24×5 = 240   │  │ #### 360       │  ✓ correct
     │ + 120 = 360    │  │                │
     │ #### 360       │  │                │
     └────────────────┘  └────────────────┘
     ✓ correct            ✓ correct (lucky guess)

     ┌────────────────┐  ┌────────────────┐
     │ Completion 3:  │  │ Completion 4:  │
     │ 24×15 = 350    │  │ The answer is  │
     │ #### 350       │  │ probably 300   │
     └────────────────┘  └────────────────┘
     ✗ wrong              ✗ wrong (no format)

  3. SCORE each completion
     rewards = [1.0, 1.0, 0.0, 0.0]

  4. COMPUTE group-relative advantage
     mean = 0.5,  std = 0.5
     advantages = [(1-0.5)/0.5, (1-0.5)/0.5, (0-0.5)/0.5, (0-0.5)/0.5]
                = [+1.0, +1.0, -1.0, -1.0]

  5. UPDATE policy
     • Reinforce tokens that led to correct answers (positive advantage)
     • Suppress tokens that led to wrong answers (negative advantage)
     • The model learns: step-by-step reasoning → correct answers
```

## The Math

### GRPO Loss Function

```
L = (1/G) Σ  (1/|o_i|) Σ  [ min( w_t · Â_i,  clip(w_t, 1-ε, 1+ε) · Â_i ) ]
         i=1           t=1
    - β · D_KL(π_θ ‖ π_ref)
```

**Where:**

| Symbol | Meaning |
|--------|--------|
| G | Group size (completions per prompt) |
| o_i | The i-th completion |
| w_t = π_θ(token_t) / π_old(token_t) | Importance ratio — how much the policy changed |
| Â_i = (r_i - μ) / σ | Group-normalized advantage |
| ε | Clip ratio (0.2) — prevents too-large updates |
| β | KL penalty weight — keeps model close to original |

### Why It Works (The Key Insight)

```
PPO approximates advantage with a learned critic:
  A(prompt, completion) = reward - V(prompt)     ← V is a neural network

GRPO approximates advantage with group statistics:
  A(prompt, completion) = (reward - mean(group)) / std(group)

For LLM training, this is a one-step problem:
  • State  = prompt
  • Action = entire completion
  • Reward = single score at the end

In this setting, the critic is just estimating E[reward | prompt].
The group mean IS a Monte Carlo estimate of E[reward | prompt].

So GRPO replaces a 7B neural network with... a mean() call.
```

### GRPO vs PPO vs DPO

| | DPO | PPO (RLHF) | GRPO |
|--|-----|-----------|------|
| **Training signal** | Fixed preference pairs | Learned reward model | Rule-based reward |
| **Generates during training?** | No (offline) | Yes (online) | Yes (online) |
| **Extra models needed** | Reference model | Reward model + Critic | Reference model (optional) |
| **Memory** | ~2x policy | ~4x policy | ~1-2x policy |
| **Best for** | Preference alignment | General RLHF | Verifiable tasks (math, code) |
| **Can discover reasoning?** | No | Yes (with effort) | **Yes (naturally)** |

---
# Part 2: Setup

## Step 1: Install & Check GPU

In [ ]:
!pip install -q transformers trl datasets accelerate peft bitsandbytes

In [ ]:
import torch, gc, time, os, re, json
import numpy as np
os.environ["WANDB_DISABLED"] = "true"

assert torch.cuda.is_available(), "GPU required!"

NUM_GPUS = torch.cuda.device_count()
for i in range(NUM_GPUS):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print("  GPU %d: %s (%.0f GB)" % (i, name, mem))
print("\nTotal GPUs: %d" % NUM_GPUS)

def gpu_report(label):
    used = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print("%s -> Allocated: %.2f GB | Reserved: %.2f GB" % (label, used, reserved))

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## Step 2: Load GSM8K — Grade School Math

7,473 math word problems with step-by-step solutions.
Each answer ends with `#### <number>` — this is our verifiable reward signal.

```
Question: Janet's ducks lay 16 eggs per day. She eats 3 for breakfast
          and uses 4 to make muffins. How much does she earn selling
          the rest at $2 each?

Answer:   16 - 3 - 4 = 9 eggs left
          9 × $2 = $18
          #### 18        ← this is what we verify
```

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main", split="train")
test_dataset = load_dataset("openai/gsm8k", "main", split="test")

print("Train: %d problems" % len(dataset))
print("Test:  %d problems" % len(test_dataset))
print("\nColumns:", dataset.column_names)
print("\n--- Example ---")
print("Question:", dataset[0]["question"][:200])
print("Answer:", dataset[0]["answer"][-100:])

In [ ]:
# Format prompts for GRPO
# GRPOTrainer needs a "prompt" column (list of chat messages)

SYSTEM_PROMPT = """Solve this math problem step by step.
Show your reasoning, then give the final answer after ####.
Example format:
Step 1: ...
Step 2: ...
#### 42"""

def format_prompt(example):
    example["prompt"] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]},
    ]
    return example

dataset = dataset.map(format_prompt)
# Keep "answer" column — it gets forwarded to reward functions as kwargs

print("Formatted %d prompts" % len(dataset))
print("\nSample prompt:")
for msg in dataset[0]["prompt"]:
    print("  [%s]: %s" % (msg["role"], msg["content"][:100]))

## Step 3: Define Reward Functions

This is the **entire training signal**. No reward model, no human labels.
Just two simple rules:

```
Reward 1 — Correctness (0 or 1):
  Extract the number after ####
  Compare with ground truth
  Correct = 1.0, Wrong = 0.0

Reward 2 — Format (0 or 0.1):
  Did the model use the #### format?
  Yes = 0.1 bonus, No = 0.0
```

The format reward is small but important — it teaches the model to structure its output
so we can actually check the answer.

In [ ]:
def extract_answer(text):
    """Extract the number after #### from a string."""
    match = re.search(r'####\s*([\d,.-]+)', text)
    if match:
        return match.group(1).replace(",", "").strip()
    return None

def correctness_reward(completions, answer, **kwargs):
    """Binary reward: 1.0 if final answer matches ground truth, 0.0 otherwise."""
    rewards = []
    for completion, gt_answer in zip(completions, answer):
        # Extract predicted answer from model output
        pred = extract_answer(completion)
        # Extract ground truth answer
        gold = extract_answer(gt_answer)
        if pred is not None and gold is not None and pred == gold:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards

def format_reward(completions, **kwargs):
    """Small bonus for using the #### answer format."""
    rewards = []
    for completion in completions:
        if re.search(r'####\s*[\d,.-]+', completion):
            rewards.append(0.1)
        else:
            rewards.append(0.0)
    return rewards

# Test the reward functions
test_completions = [
    "Step 1: 24 × 15 = 360\n#### 360",   # correct + format
    "The answer is 360",                    # correct but no format
    "Step 1: 24 × 15 = 350\n#### 350",    # wrong + format
    "I don't know",                         # wrong + no format
]
test_answers = ["#### 360"] * 4

print("Reward function test:")
cr = correctness_reward(test_completions, test_answers)
fr = format_reward(test_completions)
for i, (comp, c, f) in enumerate(zip(test_completions, cr, fr)):
    print("  [%d] %-45s correct=%.1f  format=%.1f  total=%.1f" % (i, comp[:45], c, f, c + f))

## Step 4: Load Model with QLoRA

Qwen2.5-1.5B in 4-bit — fits on T4 with room for GRPO's multiple generations.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B"

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA config for GRPO
lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

gpu_report("Before loading model")
print("Loading %s in 4-bit..." % MODEL_NAME)

---
# Part 3: GRPO Training

## The Training Loop Visualized

```
┌─────────────────────────────────────────────────────────────────┐
│                    GRPO Training Loop                          │
│                                                                 │
│  ┌──────────┐    ┌──────────────────┐    ┌───────────────┐     │
│  │  Prompt   │───>│  Generate G=4    │───>│  Score each   │     │
│  │  batch    │    │  completions     │    │  with reward  │     │
│  └──────────┘    │  per prompt      │    │  functions    │     │
│                   └──────────────────┘    └───────┬───────┘     │
│                                                    │            │
│                                                    v            │
│  ┌──────────┐    ┌──────────────────┐    ┌───────────────┐     │
│  │  Update   │<──│  Compute group   │<──│  rewards =    │     │
│  │  policy   │    │  advantages      │    │  [1,0,1,0]   │     │
│  │  weights  │    │  Â = (r-μ)/σ     │    │              │     │
│  └──────────┘    └──────────────────┘    └───────────────┘     │
│       │                                                         │
│       └─────────── repeat ──────────────────────────────────────┘
│
│  Key insight: completions compete WITHIN their group.
│  The model learns which token patterns lead to correct answers.
│  Over time, it discovers that step-by-step reasoning works best.
```

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# GRPO configuration
grpo_config = GRPOConfig(
    output_dir="./grpo_output",
    
    # GRPO-specific parameters
    num_generations=4,             # G=4 completions per prompt
    temperature=0.9,               # high temperature = diverse completions
    max_completion_length=512,     # max tokens for model's response
    max_prompt_length=256,         # max tokens for the prompt
    
    # KL penalty — keeps model close to original
    beta=0.04,                     # moderate KL penalty
    
    # Training
    num_train_epochs=1,
    per_device_train_batch_size=1, # small batch — GRPO generates G*batch sequences
    gradient_accumulation_steps=4, # effective batch = 4 prompts × 4 generations = 16 sequences
    learning_rate=5e-6,            # low LR for RL stability
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    
    # Memory
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    
    # Logging
    logging_steps=5,
    report_to="none",
    save_strategy="no",
    
    # Max steps (for demo — full training would be much longer)
    max_steps=200,
)

print("GRPO Config:")
print("  Completions per prompt: %d" % grpo_config.num_generations)
print("  Temperature: %.1f" % grpo_config.temperature)
print("  Max completion tokens: %d" % grpo_config.max_completion_length)
print("  KL penalty (beta): %.3f" % grpo_config.beta)
print("  Learning rate: %.1e" % grpo_config.learning_rate)
print("  Effective sequences per step: %d × %d = %d" % (
    grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps,
    grpo_config.num_generations,
    grpo_config.per_device_train_batch_size * grpo_config.gradient_accumulation_steps * grpo_config.num_generations,
))

In [ ]:
# Create the GRPO trainer
trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=[correctness_reward, format_reward],
    args=grpo_config,
    train_dataset=dataset,
    peft_config=lora_config,
    model_init_kwargs={
        "quantization_config": bnb_config,
        "trust_remote_code": True,
    },
)

gpu_report("After creating GRPO trainer")

In [ ]:
print("=" * 60)
print("  GRPO TRAINING: Teaching Qwen2.5-1.5B to Reason")
print("=" * 60)
print("\nThis will take a while — the model generates %d completions" % grpo_config.num_generations)
print("per prompt, scores them, then updates. Generation is the bottleneck.\n")

start = time.time()
trainer.train()
train_time = time.time() - start

gpu_report("After training")
peak_mem = torch.cuda.max_memory_allocated() / 1e9
print("\nPeak GPU: %.1f GB" % peak_mem)
print("Training time: %.0f seconds (%.1f minutes)" % (train_time, train_time / 60))

# Save
trainer.save_model("./grpo_output/final")
tokenizer.save_pretrained("./grpo_output/final")
print("Model saved.")

---
# Part 4: Evaluation — Did It Learn to Reason?

In [ ]:
# Evaluate on held-out GSM8K test set
model = trainer.model
model.eval()

# Sample 50 test problems
test_subset = test_dataset.shuffle(seed=42).select(range(50))

correct = 0
results_log = []

print("Evaluating on 50 GSM8K test problems...\n")

for i, example in enumerate(test_subset):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,     # low temp for eval (deterministic)
            do_sample=True,
        )
    
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    pred = extract_answer(response)
    gold = extract_answer(example["answer"])
    is_correct = pred is not None and gold is not None and pred == gold
    
    if is_correct:
        correct += 1
    
    results_log.append({
        "question": example["question"][:100],
        "predicted": pred,
        "gold": gold,
        "correct": is_correct,
        "response_length": len(response.split()),
        "response": response[:300],
    })
    
    if (i + 1) % 10 == 0:
        print("  %d/%d — accuracy so far: %.1f%%" % (i + 1, 50, 100 * correct / (i + 1)))

accuracy = correct / len(test_subset)
print("\n" + "=" * 50)
print("  GSM8K Test Accuracy: %d/%d = %.1f%%" % (correct, len(test_subset), accuracy * 100))
print("=" * 50)

In [ ]:
# Show some example reasoning chains
print("=" * 70)
print("  EXAMPLE REASONING CHAINS")
print("=" * 70)

# Show 3 correct and 2 incorrect
correct_examples = [r for r in results_log if r["correct"]][:3]
wrong_examples = [r for r in results_log if not r["correct"]][:2]

for label, examples in [("CORRECT", correct_examples), ("INCORRECT", wrong_examples)]:
    print("\n--- %s ANSWERS ---" % label)
    for ex in examples:
        print("\nQ: %s..." % ex["question"][:80])
        print("Model: %s" % ex["response"][:250])
        print("Predicted: %s | Gold: %s | Words: %d" % (ex["predicted"], ex["gold"], ex["response_length"]))
        print("-" * 70)

---
# Part 5: Visualizations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')

# Extract training metrics from trainer log history
log_history = trainer.state.log_history

# Parse metrics
steps, losses, rewards, comp_lengths = [], [], [], []
for entry in log_history:
    if "loss" in entry:
        steps.append(entry.get("step", 0))
        losses.append(entry["loss"])
    if "reward" in entry:
        rewards.append((entry.get("step", 0), entry["reward"]))
    if "reward/mean" in entry:
        rewards.append((entry.get("step", 0), entry["reward/mean"]))
    if "completion_length/mean" in entry:
        comp_lengths.append((entry.get("step", 0), entry["completion_length/mean"]))

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("GRPO Training — Teaching a Model to Reason", fontsize=16, color="#c9d1d9")

# 1. Training loss
if steps and losses:
    axes[0, 0].plot(steps, losses, color="#818cf8", linewidth=2, alpha=0.8)
    axes[0, 0].set_title("Training Loss", fontsize=13)
    axes[0, 0].set_xlabel("Step")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].grid(True, alpha=0.2)
else:
    axes[0, 0].text(0.5, 0.5, "Loss data not available", ha="center", va="center", color="#8b949e")
    axes[0, 0].set_title("Training Loss", fontsize=13)

# 2. Mean reward over training
if rewards:
    r_steps, r_vals = zip(*rewards)
    axes[0, 1].plot(r_steps, r_vals, color="#3fb950", linewidth=2, alpha=0.8)
    axes[0, 1].set_title("Mean Reward (should increase)", fontsize=13)
    axes[0, 1].set_xlabel("Step")
    axes[0, 1].set_ylabel("Reward")
    axes[0, 1].grid(True, alpha=0.2)
    axes[0, 1].axhline(y=0.5, color="#f0883e", linestyle="--", alpha=0.5, label="Random baseline")
    axes[0, 1].legend()
else:
    axes[0, 1].text(0.5, 0.5, "Reward data not available", ha="center", va="center", color="#8b949e")
    axes[0, 1].set_title("Mean Reward", fontsize=13)

# 3. Completion length over training (the "aha moment" signal)
if comp_lengths:
    cl_steps, cl_vals = zip(*comp_lengths)
    axes[1, 0].plot(cl_steps, cl_vals, color="#f0883e", linewidth=2, alpha=0.8)
    axes[1, 0].set_title('Completion Length ("Aha Moment" Signal)', fontsize=13)
    axes[1, 0].set_xlabel("Step")
    axes[1, 0].set_ylabel("Avg tokens")
    axes[1, 0].grid(True, alpha=0.2)
    axes[1, 0].annotate("If length increases →\nmodel is learning\nto think more",
                        xy=(0.95, 0.95), xycoords="axes fraction",
                        ha="right", va="top", fontsize=10, color="#8b949e",
                        bbox=dict(boxstyle="round", fc="#161b22", ec="#30363d"))
else:
    axes[1, 0].text(0.5, 0.5, "Completion length data not available", ha="center", va="center", color="#8b949e")
    axes[1, 0].set_title('Completion Length', fontsize=13)

# 4. Response length distribution (test set)
response_lengths = [r["response_length"] for r in results_log]
correct_lengths = [r["response_length"] for r in results_log if r["correct"]]
wrong_lengths = [r["response_length"] for r in results_log if not r["correct"]]

if correct_lengths and wrong_lengths:
    axes[1, 1].hist(correct_lengths, bins=15, alpha=0.7, color="#3fb950", label="Correct")
    axes[1, 1].hist(wrong_lengths, bins=15, alpha=0.7, color="#f87171", label="Wrong")
    axes[1, 1].set_title("Response Length: Correct vs Wrong", fontsize=13)
    axes[1, 1].set_xlabel("Words")
    axes[1, 1].set_ylabel("Count")
    axes[1, 1].legend()
    axes[1, 1].annotate("Correct answers tend\nto have more reasoning",
                        xy=(0.95, 0.95), xycoords="axes fraction",
                        ha="right", va="top", fontsize=10, color="#8b949e",
                        bbox=dict(boxstyle="round", fc="#161b22", ec="#30363d"))
elif response_lengths:
    axes[1, 1].hist(response_lengths, bins=15, alpha=0.7, color="#818cf8")
    axes[1, 1].set_title("Response Length Distribution", fontsize=13)
    axes[1, 1].set_xlabel("Words")
    axes[1, 1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# GRPO Algorithm Visualization — how advantages flow
plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title("GRPO: How Group Advantages Work", fontsize=16, color="#c9d1d9", pad=20)

# Prompt box
ax.add_patch(plt.Rectangle((0.5, 3), 2.5, 2, facecolor="#1e1b4b", edgecolor="#4338ca", linewidth=2, zorder=2))
ax.text(1.75, 4.3, "Prompt", ha="center", fontsize=12, fontweight="bold", color="#c4b5fd")
ax.text(1.75, 3.7, '"What is\n24 × 15?"', ha="center", fontsize=9, color="#8b949e")

# Completions
completions_data = [
    ("24×10+24×5\n=360  #### 360", 1.0, "#3fb950", "+1.0"),
    ("360\n#### 360", 1.0, "#3fb950", "+1.0"),
    ("24×15=350\n#### 350", 0.0, "#f87171", "-1.0"),
    ("Probably 300", 0.0, "#f87171", "-1.0"),
]

for i, (text, reward, color, adv) in enumerate(completions_data):
    y = 6.5 - i * 1.8
    # Arrow from prompt
    ax.annotate("", xy=(4, y + 0.4), xytext=(3, 4),
                arrowprops=dict(arrowstyle="->", color="#4338ca", lw=1.5))
    # Completion box
    ax.add_patch(plt.Rectangle((4, y - 0.2), 2.8, 1.0,
                facecolor="#161b22", edgecolor=color, linewidth=2, zorder=2))
    ax.text(5.4, y + 0.5, text, ha="center", fontsize=8, color="#c9d1d9", family="monospace")
    # Reward
    ax.text(7.2, y + 0.5, "r=%.1f" % reward, fontsize=10, color=color, fontweight="bold")
    # Advantage
    ax.add_patch(plt.Rectangle((8, y), 1.5, 0.8,
                facecolor=color, alpha=0.2, edgecolor=color, linewidth=1.5, zorder=2))
    ax.text(8.75, y + 0.4, "Â=%s" % adv, ha="center", fontsize=10, color=color, fontweight="bold")

# Labels
ax.text(5.4, 7.8, "Generate G=4", ha="center", fontsize=11, color="#a78bfa", fontweight="bold")
ax.text(7.2, 7.8, "Score", ha="center", fontsize=11, color="#a78bfa", fontweight="bold")
ax.text(8.75, 7.8, "Advantage", ha="center", fontsize=11, color="#a78bfa", fontweight="bold")

# Group stats annotation
ax.text(8.75, 0.3, "μ=0.5  σ=0.5", ha="center", fontsize=9, color="#8b949e",
        bbox=dict(boxstyle="round", fc="#161b22", ec="#30363d"))

plt.tight_layout()
plt.show()

## Results Summary

In [ ]:
from IPython.display import HTML, display

avg_correct_len = np.mean(correct_lengths) if correct_lengths else 0
avg_wrong_len = np.mean(wrong_lengths) if wrong_lengths else 0

html = """
<div style="font-family:-apple-system,sans-serif;max-width:820px;margin:20px 0;">
  <h3 style="color:#c9d1d9;">GRPO Training Results</h3>

  <div style="display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:16px;">
    <div style="background:linear-gradient(135deg,#1a2332,#161b22);border:1px solid #30363d;
                border-radius:12px;padding:18px;text-align:center;">
      <div style="color:#a78bfa;font-size:12px;font-weight:600;">GSM8K Accuracy</div>
      <div style="color:#3fb950;font-size:28px;font-weight:700;margin:6px 0;">%.1f%%%%</div>
      <div style="color:#8b949e;font-size:11px;">%d / %d correct</div>
    </div>
    <div style="background:linear-gradient(135deg,#1a2332,#161b22);border:1px solid #30363d;
                border-radius:12px;padding:18px;text-align:center;">
      <div style="color:#a78bfa;font-size:12px;font-weight:600;">Training Time</div>
      <div style="color:#58a6ff;font-size:28px;font-weight:700;margin:6px 0;">%.0fm</div>
      <div style="color:#8b949e;font-size:11px;">%d steps</div>
    </div>
    <div style="background:linear-gradient(135deg,#1a2332,#161b22);border:1px solid #30363d;
                border-radius:12px;padding:18px;text-align:center;">
      <div style="color:#a78bfa;font-size:12px;font-weight:600;">Peak GPU</div>
      <div style="color:#f0883e;font-size:28px;font-weight:700;margin:6px 0;">%.1f GB</div>
      <div style="color:#8b949e;font-size:11px;">QLoRA + GRPO gen</div>
    </div>
    <div style="background:linear-gradient(135deg,#1a2332,#161b22);border:1px solid #30363d;
                border-radius:12px;padding:18px;text-align:center;">
      <div style="color:#a78bfa;font-size:12px;font-weight:600;">Avg Response</div>
      <div style="color:#d2a8ff;font-size:28px;font-weight:700;margin:6px 0;">%d</div>
      <div style="color:#8b949e;font-size:11px;">words per answer</div>
    </div>
  </div>

  <div style="background:#161b22;border:1px solid #30363d;border-radius:12px;padding:18px;">
    <table style="width:100%%;color:#c9d1d9;font-size:13px;border-spacing:0 8px;">
      <tr><td style="color:#8b949e;">Algorithm</td><td style="text-align:right;color:#a78bfa;font-weight:600;">GRPO (Group Relative Policy Optimization)</td></tr>
      <tr><td style="color:#8b949e;">Model</td><td style="text-align:right;">Qwen2.5-1.5B (4-bit QLoRA)</td></tr>
      <tr><td style="color:#8b949e;">Dataset</td><td style="text-align:right;">GSM8K (7,473 math problems)</td></tr>
      <tr><td style="color:#8b949e;">Reward</td><td style="text-align:right;">Binary accuracy + format bonus (no reward model)</td></tr>
      <tr><td style="color:#8b949e;">Completions per prompt</td><td style="text-align:right;">%d</td></tr>
      <tr><td style="color:#8b949e;">Avg length (correct)</td><td style="text-align:right;color:#3fb950;">%.0f words</td></tr>
      <tr><td style="color:#8b949e;">Avg length (wrong)</td><td style="text-align:right;color:#f87171;">%.0f words</td></tr>
    </table>
  </div>
</div>
""" % (
    accuracy * 100, correct, len(test_subset),
    train_time / 60, grpo_config.max_steps,
    peak_mem,
    int(np.mean(response_lengths)) if response_lengths else 0,
    grpo_config.num_generations,
    avg_correct_len,
    avg_wrong_len,
)

display(HTML(html))

---
# Part 6: Deep Dive — Everything About GRPO

## The "Aha Moment"

During DeepSeek-R1-Zero training (pure RL, no SFT), the model spontaneously developed
self-reflective reasoning. An intermediate checkpoint produced:

> *"Wait, wait. Wait. That's an aha moment I can flag here."*

The model learned to:
- Allocate more thinking time to harder problems
- Re-evaluate approaches mid-reasoning
- Self-correct by catching its own mistakes

**None of this was explicitly trained.** The only signal was: "is the final answer correct?"

### How does reasoning emerge from a simple reward?

```
Within each group of G completions for the same prompt:

  Completion A: "24 × 15 = 360. #### 360"          reward = 1.0  ✓
  Completion B: "Let me think... 24×10=240,          reward = 1.0  ✓
                 24×5=120, 240+120=360. #### 360"
  Completion C: "24 × 15 = 350. #### 350"           reward = 0.0  ✗
  Completion D: "Hmm, maybe 300?"                    reward = 0.0  ✗

Advantage: A=+1, B=+1, C=-1, D=-1

Both A and B are reinforced. But as training progresses and
problems get harder, only completions with careful reasoning
(like B) consistently get correct answers.

The group mean rises → the bar for positive advantage rises →
the model must develop increasingly sophisticated reasoning
to stand out within its own group.

This creates a natural curriculum without any explicit design.
```

## GRPO vs PPO — The Full Comparison

### Memory
```
PPO:  policy(7B) + critic(7B) + ref_model(7B) + reward_model(7B) = 28B params
GRPO: policy(7B) + ref_model(7B, optional)                       = 7-14B params
```

### Advantage Estimation
```
PPO:   A = R - V(s)           ← V is a neural network (7B params)
GRPO:  A = (r - mean) / std   ← just arithmetic on group rewards

Why this works:
LLM training is a one-step problem (bandit):
  State  = prompt (fixed)
  Action = entire completion (one shot)
  Reward = single score at the end

In this setting, V(s) ≈ E[reward | prompt].
The group mean IS a Monte Carlo estimate of E[reward | prompt].
So GRPO replaces a 7B neural network with a mean() call.
```

### The KL Penalty
```
Without KL penalty:
  Model drifts far from pretrained weights
  → "reward hacking" (exploits reward function quirks)
  → loses general language ability

With KL penalty (β=0.04):
  D_KL(π_θ ‖ π_ref) = Σ [ π_ref/π_θ - log(π_ref/π_θ) - 1 ]
  → gently pulls model back toward original behavior
  → model improves at math while staying coherent
```

## When to Use GRPO vs DPO vs PPO

| Scenario | Best method | Why |
|----------|------------|-----|
| Math reasoning | **GRPO** | Verifiable reward (correct/incorrect) |
| Code generation | **GRPO** | Verifiable reward (tests pass/fail) |
| General chat quality | **DPO** | No clear right/wrong, need preferences |
| Safety alignment | **PPO** or DPO | Complex reward landscape |
| Instruction following | **DPO** | Preference data readily available |
| Any task with ground truth | **GRPO** | Simple, stable, no reward model |

## Scaling Up

| Scale | Setup | Expected GSM8K |
|-------|-------|----------------|
| This notebook | 1.5B, 200 steps, T4 | ~15-25% |
| Serious training | 7B, 2000 steps, A100 | ~40-55% |
| DeepSeek-R1-Zero | 671B MoE, full training | ~86% |
| DeepSeek-R1 (SFT+RL) | 671B MoE, multi-stage | ~95% |

### Key Hyperparameters to Scale

```
More compute → increase num_generations (4 → 8 → 16)
              → increase max_completion_length (512 → 1024 → 2048)
              → decrease learning_rate (5e-6 → 1e-6)
              → train for more steps (200 → 2000 → 10000)

The model needs enough generations per prompt to get
meaningful variance in rewards (some correct, some wrong).
If all G completions are correct → advantage = 0 → no learning.
If all G completions are wrong  → advantage = 0 → no learning.
```

| Platform | GPU | Max model for GRPO | Cost |
|----------|-----|-------------------|------|
| **Kaggle** | 2x T4 (30GB) | 1.5B (QLoRA) | Free |
| **Colab** | T4 (15GB) | 1.5B (QLoRA) | Free |
| **Colab Pro** | A100 (40GB) | 7B (QLoRA) | ~$10/mo |
| **Lambda** | A100 (80GB) | 14B (QLoRA) | ~$1/hr |